# 🚀 Industrial Time-Series Intelligence Platform — Google Colab GPU Training
### **High-Accuracy Multivariate Forecasting (Transformer, LSTM, GRU) & Dual Anomaly Detection**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anurag-snippet/TimeSeriesGPT-Transformer-Based-Multivariate-Forecasting/blob/main/notebooks/TimeSeriesGPT_Colab_Training.ipynb)

---
### 📌 Objectives
1. **Full GPU Convergence**: Train all deep-learning forecasters (**Transformer Encoder**, **LSTM**, **GRU**) on full telemetry dataset (8,000+ sequences) across 35 epochs with Early Stopping and Learning Rate Decay.
2. **Physical Sensor Evaluation**: Measure MAE, RMSE, MAPE, and sMAPE in original physical engineering units (Kelvin, psia, RPM, g-force).
3. **Dual Anomaly Detection**: Evaluate LSTM Autoencoder vs. Isolation Forest with synthetic fault injection.
4. **One-Click Export**: Download all checkpoints and prediction artifacts to sync directly with the local Streamlit dashboard.

## 1. Verify GPU Hardware Acceleration
Colab provides free access to NVIDIA T4 / A100 GPUs. Ensure your runtime type is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f" Active GPU Detected: {gpus[0].name}")
    !nvidia-smi
else:
    print("⚠️ No GPU detected. Go to Runtime -> Change runtime type -> Select T4 GPU for faster training.")

## 2. Clone Repository & Install Dependencies

In [ ]:
import os

# Clone the repository if not already in directory
if not os.path.exists("src"):
    !git clone https://github.com/Anurag-snippet/TimeSeriesGPT-Transformer-Based-Multivariate-Forecasting.git TimeSeriesGPT
    %cd TimeSeriesGPT
else:
    print("Already in repository workspace.")

# Install project dependencies
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
print(" Dependencies installed successfully!")

## 3. Prepare Industrial Sensor Telemetry
- Generates 12,000 timesteps of 7-channel turbine sensors (`T24`, `T30`, `T50`, `P30`, `Nf`, `Nc`, `vib`).
- Chronological train/val/test split (70% / 15% / 15%).
- Fits `StandardScaler` **strictly** on training split to eliminate data leakage.

In [ ]:
!python scripts/prepare_data.py

## 4. Train Classical Empirical Baselines
Establishes realistic performance floors for comparison: Naive Persistence, Moving Average (window=10), and Multi-Output Ridge Regression.

In [ ]:
!python scripts/train.py --model naive
!python scripts/train.py --model moving_average
!python scripts/train.py --model ridge

## 5. Train Deep Learning Models on GPU (Full Convergence)
Executes full multi-step training (60-timestep input window, 10-timestep forecast horizon) with:
- **Transformer Encoder**: Positional Encodings + MultiHeadAttention + Pre-LN + Feed-Forward.
- **Stacked LSTM**: 2-layer recurrent architecture with dropout.
- **Stacked GRU**: Gated recurrent baseline with lower parameter footprint.
- Automatic `EarlyStopping(patience=7)` and `ReduceLROnPlateau(patience=3)`.

In [ ]:
# 1. Transformer Forecaster
print("=== Training Transformer Forecaster ===")
!python scripts/train.py --model transformer --epochs 35

# 2. Stacked LSTM Forecaster
print("\n=== Training Stacked LSTM Forecaster ===")
!python scripts/train.py --model lstm --epochs 35

# 3. Stacked GRU Forecaster
print("\n=== Training Stacked GRU Forecaster ===")
!python scripts/train.py --model gru --epochs 35

## 6. Run Systematic Benchmark & Anomaly Detection
- Evaluates test metrics across all models.
- Injects synthetic sensor anomalies (thermal drift, sudden pressure drop, sensor freeze) to evaluate **LSTM Autoencoder** vs. **Isolation Forest** (Precision, Recall, F1, ROC-AUC).

In [ ]:
!python scripts/evaluate.py

## 7. Inspect Benchmark Results & Visual Forecasts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load results table
results_df = pd.read_csv("experiments/results.csv")
display(results_df[["model", "MAE", "RMSE", "MAPE", "SMAPE", "training_time", "parameter_count"]])

# 2. Plot MAE and sMAPE comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
models = results_df["model"]

ax1.bar(models, results_df["MAE"], color="#1f77b4", edgecolor="black", alpha=0.85)
ax1.set_title("Mean Absolute Error (Physical Sensor Scale)", fontweight="bold")
ax1.set_ylabel("MAE")
ax1.grid(axis="y", linestyle="--", alpha=0.5)
ax1.tick_params(axis="x", rotation=25)

ax2.bar(models, results_df["SMAPE"], color="#ff7f0e", edgecolor="black", alpha=0.85)
ax2.set_title("Symmetric MAPE (% Error)", fontweight="bold")
ax2.set_ylabel("sMAPE (%)")
ax2.grid(axis="y", linestyle="--", alpha=0.5)
ax2.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

# 3. Plot sample Transformer multi-step forecast
pred_data = np.load("artifacts/predictions/transformer_sample_prediction.npz")
hist = pred_data["history"][0]
true_fut = pred_data["ground_truth"][0]
pred_fut = pred_data["prediction"][0]
sensor_names = pred_data["feature_names"]

plt.figure(figsize=(12, 5))
sensor_idx = 0  # T24 Combustor Inlet Temp
t_hist = np.arange(len(hist))
t_fut = np.arange(len(hist), len(hist) + len(true_fut))

plt.plot(t_hist, hist[:, sensor_idx], label="Historical Input (60 steps)", color="#333333")
plt.plot(t_fut, true_fut[:, sensor_idx], label="Ground Truth Future", color="#2ca02c", marker="o", markersize=4)
plt.plot(t_fut, pred_fut[:, sensor_idx], label="Transformer Prediction", color="#d62728", linestyle="--", marker="x", markersize=5)
plt.axvline(x=len(hist)-1, color="gray", linestyle=":", label="Forecast Horizon Cutoff")
plt.title(f"Transformer Multi-Step Forecast: {sensor_names[sensor_idx]}", fontweight="bold")
plt.xlabel("Time Steps")
plt.ylabel("Sensor Value (Physical Units)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Download Trained Artifacts & Checkpoints to Local Machine
This compresses `artifacts/` (model weights, scalers, prediction arrays) and `experiments/results.csv` into a zip file and automatically initiates the browser download.

**Next step on your local machine:**
1. Unzip `timeseriesgpt_colab_artifacts.zip` directly into your local `TimeSeriesGPT` project directory.
2. Run `streamlit run app.py` to view your high-accuracy models in the interactive platform!

In [ ]:
from google.colab import files

# Package checkpoints, predictions, scalers, and experimental benchmark results
!zip -r timeseriesgpt_colab_artifacts.zip artifacts/ experiments/results.csv

print(" Downloading trained artifacts zip archive...")
files.download("timeseriesgpt_colab_artifacts.zip")